In [ ]:
import xarray as xr
import matplotlib.pyplot as plt
from workflow.scripts.utils import global_avg
import matplotlib as mpl
import pandas as pd
import numpy as np
from workflow.scripts.utils import t_test_diff_sample_means, get_forcing
from workflow.scripts.plotting_tools import get_model_colordict
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

In [ ]:
order_forcings = [   
            
            'GFDL-ESM4',
            'CNRM-ESM2-1',
            'UKESM1-0-LL',
            'IPSL-CM6A-LR-INCA',
            'NorESM2-LM',
            'MPI-ESM-1-2-HAM',
            'EC-Earth3-AerChem'
            
        ]

order= [   
            'GISS-E2-1-G',
            'MIROC6',
            'GFDL-ESM4',
            'CNRM-ESM2-1',
            'UKESM1-0-LL',
            'IPSL-CM6A-LR-INCA',
            'NorESM2-LM',
            'MPI-ESM-1-2-HAM',
            'EC-Earth3-AerChem'
            
        ]

ds_exp = {p.split("_")[-2]: xr.open_dataset(p).isel(time=slice(1,None)) for p in snakemake.input.exp_data}
ds_ctrl = {p.split("_")[-2]: xr.open_dataset(p) for p in snakemake.input.ctrl_data}

conf_level = snakemake.params.get('CI_alpha', 0.05)

variable_order = ['lwp', 'clivi', 'cl_low', 'cl_middle', 'cl_high', 'clt','cdncvi', 'pr']

In [ ]:
dfs_forcing = {p.split('.')[0].split('_')[-1]: pd.read_csv(p,index_col=0) for p in snakemake.input.forcing_tables}
colors = get_model_colordict()
model_order = snakemake.params.get('model_order', order)

In [ ]:
def setup_plot_abs_Forcing(ax):
    ax.set_xlim(-1, 1)
    # ax.set_ylim(-0.25, 9.1)
    ax.xaxis.set_major_locator(mpl.ticker.FixedLocator([-0.8, -0.6, -0.4, -0.2, 0, 0.2,0.4, 0.6,0.8]))
    ax.set_yticks([])
    ax.xaxis.set_minor_locator(mpl.ticker.AutoMinorLocator(4))
    ax.axvline(0.0, linestyle=":", linewidth=2, color="darkgrey")
    ax.tick_params(top=True, which="both", labeltop=True)
    ax.spines["left"].set_visible(True)
    ax.spines["right"].set_visible(True)
    ax.set_xlabel("W m-2")
    ax.set_title('Dust Cloud Forcing')
    ax.text(0.98, 0.85, "LW", va="center", ha="right", transform=ax.transAxes, fontsize=10)
    ax.text(0.98, 0.5, "SW", va="center", ha="right", transform=ax.transAxes, fontsize=10)
    ax.text(0.98, 0.18, "Net", va="center", ha="right", transform=ax.transAxes, fontsize=10)
    return ax

In [ ]:

def plot_forcings_bar(df, nmodels, pos0, ax,dist=.9, spacing_frac=.01, scaling: pd.Series=None
                      ,model_order: list = None):
    dx = dist/nmodels
    
    if scaling is not None:
        sign_diff = df['diff_sigificant'].copy()
        df = df.divide(scaling, axis=0)
        df['diff_sigificant'] = sign_diff
    if model_order:
        df = df.loc[order]
    else:
        df = df.sort_values('diff')
    
    pos=pos0
    gap =  spacing_frac/dist
    n=0
    for model, series in df.iterrows():
        if series.isnull()['diff']:
            continue
        else:
            if series['diff_sigificant'] == True:
                hatch='\\\\'
            else:
                hatch=None
            ax.barh(pos,series['diff'],height=dx-gap,zorder=100, facecolor=colors[model], 
                        xerr=series['pooled_std'],capsize=2, hatch=hatch)
            ax.plot(series['diff'],pos,  marker=".",  markerfacecolor=colors[model],
                    markeredgecolor= 'k',ms=10, zorder=300)
            
            if series['diff'] is not np.nan:
                n+=1
            pos+=dx
    mean = df.mean(axis=0)
    ax.plot(mean['diff'],pos0+dist/(n/2), mfc='#FF005E',linestyle='', 
                    marker='*', ms = 12, zorder=301, markeredgecolor='k')
        

In [ ]:
def _fill_diag_df(data,df, variables):
    
    for variable in variables:
        temp_data = data.get(variable)
        if temp_data is None:
            df[variable] = np.nan
        else:
            df[variable] = temp_data.values
    return df

def calc_check_diff(ctrl,exp, ci):
    diff = xr.zeros_like(exp.isel(time=0).mean(dim=['lon','lat']))
    sig_ds = xr.zeros_like(diff)
    for dvar in exp.data_vars:
        t, p, d = t_test_diff_sample_means(da_ctrl =ctrl[dvar], 
                                            da_exp=exp[dvar],global_mean=True)
        
        diff = diff.assign({dvar:d})
        sig_ds = sig_ds.assign({dvar:p < ci})
    return diff, sig_ds

def create_diagnostics_df(ds_ctrl, ds_exp, mod_id,
                        variables=['lwp','pr','cl_low','cl_middle','cl_high','cdncvi','clt','clivi'],
                          ci=0.05
                        ):
    ctrl = ds_ctrl.mean(dim='time')
    exp = ds_exp.mean(dim='time')

    if np.all(ctrl.lat.values == exp.lat.values):
        diff, sig = calc_check_diff(ds_ctrl, ds_exp, ci)
    else:
        ds_exp = ds_exp.assign_coords(lat=ds_ctrl.lat.values)
        diff, sig = calc_check_diff(ds_ctrl, ds_exp,ci)
        
    
    ctrl = global_avg(ctrl)
    exp = global_avg(exp)
    
#     print(variables)

    series_exp = pd.Series(index=variables,name=mod_id)
    series_ctrl = pd.Series(index=variables,name=mod_id)
    series_diff = pd.Series(index=variables,name=mod_id)
    series_sig = pd.Series(index=variables,name=mod_id)
    reldiff = pd.Series(index=variables,name=mod_id)

    series_exp = _fill_diag_df(exp,series_exp,variables)
    series_ctrl = _fill_diag_df(ctrl,series_ctrl,variables)
    series_diff = _fill_diag_df(diff,series_diff,variables)
    series_sig = _fill_diag_df(sig,series_sig,variables)

    for variable in variables:

        if ctrl.get(variable) is not None:
            reldiff[variable] = diff[variable].values/ctrl[variable].values*100

    return series_exp,series_ctrl, reldiff, series_diff, series_sig


In [ ]:
def _get_fmt(data):
    if abs(data) > 100:
        valfmt_temp = mpl.ticker.StrMethodFormatter("{x:.0f}")
    elif abs(data) > 1:
        valfmt_temp = mpl.ticker.StrMethodFormatter("{x:.1f}")
    elif abs(data) < 0.3 and  abs(data) > 0.0007:
        valfmt_temp = mpl.ticker.StrMethodFormatter("{x:.3f}")
    elif abs(data) < 0.0007:
        valfmt_temp = mpl.ticker.StrMethodFormatter("{x:.2e}")
        
    else:
        valfmt_temp = mpl.ticker.StrMethodFormatter("{x:.2f}")
            # print(data[i,j],data[i,j] is np.nan)
    return valfmt_temp(data)
def annotate_heatmap(im,data, rel_change=None,sig_df=None,valfmt="{x:.2f}", 
                     textcolors=["black", "white"], threshold=3, **textkw):
    """
    A function to annotate a heatmap.
    """
    # Normalize the threshold to the images color range.

    # Set default alignment to center, but allow it to be
    # overwritten by textkw.
    kw = dict(horizontalalignment="center",verticalalignment="center")
    kw.update(textkw)
    # Get the formatter in case a string is supplied
    if isinstance(valfmt, str):
        valfmt = mpl.ticker.StrMethodFormatter(valfmt)
    # Loop over the data and create a `Text` for each "pixel".
    # Change the text's color depending on the data.
    texts = []
    cdata = im.get_array().data



    texts = []
    for i in range(data.shape[0]):
        for j in range(data.shape[1]):
            if sig_df is not None:
                if sig_df[i,j] == True:
                    kw.update(weight='bold')
                else:
                    kw.update(weight='light')
            kw.update(color=textcolors[int(abs(cdata[i, j]) > threshold)], )
            if data[i,j] > 100:
                valfmt_temp = mpl.ticker.StrMethodFormatter("{x:.0f}")
            elif data[i,j] > 1:
                valfmt_temp = mpl.ticker.StrMethodFormatter("{x:.1f}")
            elif data[i,j] < 0.3:
                valfmt_temp = mpl.ticker.StrMethodFormatter("{x:.3f}")
            else:
                valfmt_temp = mpl.ticker.StrMethodFormatter("{x:.2f}")
            # print(data[i,j],data[i,j] is np.nan)
            if np.isnan(data[i,j]):
                texts.append('')
            else:
                
                if rel_change is not None:
                    text = im.axes.text(j, i, f"{_get_fmt(data[i, j])}\n ({_get_fmt(rel_change[i, j])} %)", **kw)
                else:
                    text = im.axes.text(j, i, f"{_get_fmt(data[i, j])}", **kw)
            texts.append(text)

    return texts

In [ ]:
dfs = []
rel_dfs = []
dfs_ctrl =  []
dfs_diff = []
dfs_sig = []
for mod_id in ds_exp:
    exp,ctrl,temp_rel,diff, sig = create_diagnostics_df(ds_ctrl[mod_id], ds_exp[mod_id], mod_id,ci=conf_level)
    dfs.append(exp)
    rel_dfs.append(temp_rel)
    dfs_ctrl.append(ctrl)
    dfs_diff.append(diff)
    dfs_sig.append(sig)

In [ ]:
translate_column_names = {
    'lwp': {'name':'LWP \n (g m$^{-2}$)',
            'scale':1e3},
    'clivi': {'name':'IWP \n (g m$^{-2}$)',
            'scale': 1e3},
    'pr': {'name':'Precip \n (mm year$^{-1}$)',
            'scale': 1},
    'cl_low': {'name':'$\mathrm{CldFrac}_{low}$ \n [%]',
            'scale': 1},
    'cl_middle': {'name':'$\mathrm{CldFrac}_{mid}$ \n (%)',
            'scale': 1},
    'cl_high': {'name':'$\mathrm{CldFrac}_{high}$ \n (%)', 
            'scale': 1},
    'cdncvi': {'name':'$\mathrm{N_d}/1000$ \n (cm$^{-2}$)',
            'scale': 0.00000001},
    'clt': {'name':'CldFrac \n (-)',
            'scale': 1}

}

In [ ]:

df = pd.DataFrame(dfs).sort_index()
df_rel = pd.DataFrame(rel_dfs).sort_index()
df_ctrl = pd.DataFrame(dfs_ctrl).sort_index()
df_diff = pd.DataFrame(dfs_diff).sort_index()
df_sig = pd.DataFrame(dfs_sig).sort_index()

In [ ]:
vis_df = df_diff.copy()
vis_rel = df_rel.copy()


if model_order:
    vis_df = vis_df.loc[model_order[::-1]]
    vis_rel = vis_rel.loc[model_order[::-1]]
    df_sig = df_sig.loc[model_order[::-1]]
# vis_df.loc['UKESM1-0-LL',['cl_low','cl_middle','cl_high']] = np.nan
if variable_order:
    vis_df = vis_df[variable_order]
    vis_rel = vis_rel[variable_order]

for var in translate_column_names:
    vis_df[var] = vis_df[var]*translate_column_names[var]['scale']
    vis_df = vis_df.rename(columns={var:translate_column_names[var]['name']})
    vis_rel = vis_rel.rename(columns={var:translate_column_names[var]['name']})


In [ ]:

    
context_dict = {
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'legend.fontsize': 12,
    'figure.figsize': (8.3, 8)
}

with mpl.rc_context(context_dict):
    # Create the combined figure with GridSpec
    fig = plt.figure(figsize=(8.3, 9))  # Adjust the height to accommodate both subplots
    # Adjust the height ratios of outer_gs
    outer_gs = fig.add_gridspec(2, 1, height_ratios=[1, 2])  # This gives Plot 1 ~1/3 and Plot 2 ~2/3 of the height

    # Plot 1
    inner_gs1 = outer_gs[0].subgridspec(1, 1, )
    ax1 = fig.add_subplot(inner_gs1[0])
    dist = 1.8

    setup_plot_abs_Forcing(ax1)
    plot_forcings_bar(get_forcing('CloudEff', dfs_forcing), len(dfs_forcing), 0.02, ax1, dist=dist, model_order=order_forcings)
    plot_forcings_bar(get_forcing('SWCloudEff', dfs_forcing), len(dfs_forcing), 2, ax1, dist=dist, model_order=order_forcings)
    plot_forcings_bar(get_forcing('LWCloudEff', dfs_forcing), len(dfs_forcing), 4.1, ax1, dist=dist, model_order=order_forcings)

    legments = [
        Line2D([0], [0], markerfacecolor=colors[m], marker="o", label=m, color="w", markersize=10) for m in order_forcings[::-1]
    ]

    legments.append(
        Line2D(
            [0],
            [0],
            markerfacecolor="#FF005E",
            marker="*",
            label="Model mean",
            color="w",
            markeredgecolor="k",
            markersize=12,
        )
    )

    fig.legend(handles=legments, ncol=1, bbox_to_anchor=[0.105, 0.68, 0.5, 0.5], loc="lower left",  fontsize=8)
    # Plot 2
    inner_gs2 = outer_gs[1].subgridspec(2, 1, height_ratios=[9, 1])
    ax2 = fig.add_subplot(inner_gs2[0], aspect='equal', frameon=False)

    ax2.grid(color='w', linestyle='-', linewidth=3, which='minor')
    ax2.set_xticks(np.arange(vis_df.shape[1] + 1) - .5, minor=True)
    ax2.set_yticks(np.arange(vis_df.shape[0] + 1) - .5, minor=True)

    ax2.spines[:].set_visible(False)

    cmap = mpl.colormaps.get_cmap('PiYG').resampled(9)
    cmap.set_bad("#E6E6E6")
    im = ax2.imshow(vis_rel, cmap=cmap, aspect='auto', vmin=-2.2, vmax=2.2)
    cbar = fig.colorbar(im, ax=ax2, location='right', pad=0.06, shrink=0.8, extend='both')
    cbar.ax.set_yticks([-2, -1.5, -1, -0.5, 0, 0.5, 1, 1.5, 2])
    cbar.ax.set_ylabel('Relative change [%]')
    cbar.ax.set_position([0.78, 0.258, 0.03, 0.276])
    
    ax2.set_xticks(np.arange(vis_df.shape[1]), labels=vis_df.columns, fontsize=8)
    ax2.xaxis.tick_top()
    ax2.set_yticks(np.arange(vis_df.shape[0]), labels=vis_df.index, fontsize=8)
    ax2.tick_params(which="minor", bottom=False, left=False, top=False)
    
    texts = annotate_heatmap(im, data=vis_df.values, sig_df=df_sig.values, threshold=1.8, fontsize=7.5)
    mean_vals = vis_df.mean(axis=0)
    mean_vals = [_get_fmt(v) for v in mean_vals.values]
    
    ax3 = fig.add_subplot(inner_gs2[1], frameon=False)
    tab = ax3.table([mean_vals], edges='horizontal', loc='top', bbox=[0.05, 1.8, 0.79, 1],
                    cellLoc='center', rowLabels=['$Ens_{mean}$'])
    ax3.axis('off')
    tab.scale(xscale=.8, yscale=1.5)
    pos2 = ax1.get_position()  # position of ax2
    # pos3 = ax3.get_position()  # position of ax
    ax1.set_position([pos2.x0+0.18, pos2.y0, pos2.width-0.25, pos2.height]) 
    pos3 = ax2.get_position()
    ax2.set_position([pos3.x0+0.05, pos3.y0 + 0.05, pos3.width-0.02, pos3.height-0.075])     
    ax1.text(-0.1, 2.2, 'a)', transform=ax2.transAxes, fontsize=12, fontweight='bold', va='top', ha='right')
    ax2.text(-0.1, 1.1, 'b)', transform=ax2.transAxes, fontsize=12, fontweight='bold', va='top', ha='right')
    plt.savefig(snakemake.output.outpath, dpi=300, bbox_inches='tight')

    plt.show()